# 03 Decode-Time Training: ESamp

**Scenario**: A growing family of algorithms **learns while generating**: each request carries a small set of fast weights, updates them at every generated token using a signal from that token, and lets the updated weights shape subsequent tokens. Many test-time training variants follow this exact pattern. Here we use **ESamp** as our example:

> Each request trains a small network (a distiller) online to predict the final-layer hidden state from layer 0's hidden state. During sampling, logits computed from that shallow prediction are subtracted from the model's logits with strength `beta`: tokens already predictable from shallow features are suppressed, while tokens where deeper layers add new information are boosted.

Implementing this by hand inside an inference engine is painful: autograd must run inside inference mode; each request needs its own parameters and AdamW optimizer state allocated and reclaimed as requests arrive and finish; backward and optimizer steps must overlap with the next forward pass or throughput collapses by half; and all of it still needs to stay inside CUDA graphs.

Here, all you write is the small network (a plain `nn.Module`), one loss function, and one prediction method. Then we measure two things across 128 concurrent requests: whether sampling diversity changes, and how much throughput remains.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

# We benchmark throughput on 7B: on 0.5B a single decode step takes only a few milliseconds, which exaggerates fixed overhead; 7B is the
# typical deployment scale for this class of algorithms. bf16 7B weights take ~15 GiB, requiring a 24 GiB GPU.
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# 128 concurrent requests (8 prompts x 16 samples each).
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=128, max_num_batched_tokens=8192,
                          gpu_memory_utilization=0.85, seed=7)

## Step 1: The Small Network Is Plain PyTorch

A low-rank residual block followed by layer norm. `up` is zero-initialized, so before training begins it acts as a plain layer norm over the input.

In [2]:
class Distiller(nn.Module):
    def __init__(self, width: int, rank: int) -> None:
        super().__init__()
        self.down = nn.Linear(width, rank, bias=False)
        self.up = nn.Linear(rank, width, bias=False)
        nn.init.zeros_(self.up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = x + self.up(F.silu(self.down(x)))
        return F.layer_norm(y, y.shape[-1:])

## Step 2: The Workload

- `distiller` is a `pllm.RequestState`: **one copy per request**. `init=pllm.FromModule(...)` tells pLLM how to build the network (`m` supplies dimensions known only after model loading; `m.hidden_size` is 3584 for this 7B model), its parameters become the state, and `optimizer=` specifies how to update them. When a request finishes, both its weights and AdamW moments are reclaimed automatically.
- `loss` is decorated with `@pllm.train(state="distiller")`: it returns **one loss per row** (one row per active request token). The framework masks padding rows, reduces losses (`reduction="mean"` averages within each request state first), runs backward, applies AdamW, and publishes a new version. `pllm.module("distiller")` gives you a module where each row runs against its own request's parameters.
- `predict` is decorated with `@pllm.guide_sampling(beta=...)`: the returned hidden representation is projected through the LM head, turning the sampling distribution into `(1 + beta) * logits - beta * predicted_logits`. `rows="trained"` leaves untrained requests untouched; the `trained` selector reads how many times each request's state has been updated via `pllm.version`.

In [3]:
@dataclass(frozen=True)
class MiniESamp(pllm.Workload):
    beta: float
    rank: int = 4

    @property
    def distiller(self) -> pllm.RequestState:
        return pllm.RequestState(
            init=pllm.FromModule(lambda m: Distiller(m.hidden_size, self.rank)), optimizer=pllm.AdamW(1e-3)
        )

    def trained(self, versions: torch.Tensor = pllm.version("distiller")) -> torch.Tensor:
        return versions > 0

    @pllm.train(state="distiller")
    def loss(self, shallow: torch.Tensor = pllm.hidden(0), deep: torch.Tensor = pllm.hidden(-1),
             f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return (f(shallow) - deep).square().mean(-1)

    def strength(self) -> float:
        return self.beta

    @pllm.guide_sampling(beta=strength, rows="trained")
    def predict(self, shallow: torch.Tensor = pllm.hidden(0), f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return f(shallow)

Because the workload is a frozen dataclass with `beta` and `rank` as fields, `beta=strength` delegates the decorator argument to the method defined right above it. pLLM calls `strength()` once when binding the workload instance to read that instance's `beta`, allowing multiple instances of the same class to carry different `beta` values. Your IDE can jump directly from `strength` to the method definition, and type checkers verify it returns `float`. Avoid writing `beta=beta` directly inside the class body—that binds the class default rather than the instance value.

## Step 3: 128 Requests—Throughput and Outputs

Eight prompts, 16 continuations each, 256 tokens per sample. Three configurations:

- **Plain vLLM**: no workload;
- **ESamp, beta = 0**: each request trains its distiller normally, without guiding sampling. Training alone should not change a single sampled token;
- **ESamp, beta = 0.5**: training plus sampling guidance.

Each configuration generates once to record outputs using the same random seed so samples can be compared one-for-one. Throughput is measured via `pllm.benchmark.throughput`: one warmup run followed by three timed runs taking the median, timing full `generate` calls (including prefill) divided by actual generated tokens.

In [4]:
prompts = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)


def samples(llm) -> list[str]:
    return [sample.text for output in llm.generate(prompts, sampling, use_tqdm=False) for sample in output.outputs]


with pllm.make_llm(CONFIG) as plain:
    plain_texts = samples(plain)
    plain_rate = throughput(plain, prompts, sampling)
print(f"原生 vLLM        : {plain_rate:6.0f} token/s")

for beta in (0.0, 0.5):
    with pllm.make_llm(CONFIG, workload=MiniESamp(beta)) as llm:
        texts = samples(llm)
        rate = throughput(llm, prompts, sampling)
        report = llm.diagnostics()
    same = sum(a == b for a, b in zip(texts, plain_texts))
    print(f"ESamp beta={beta:<4}: {rate:6.0f} token/s，为原生的 {rate / plain_rate:.0%}；"
          f"{same}/{len(texts)} 个样本与原生相同；累计训练 {report.trained_rows} 行；"
          f"eager 原因 {report.eager_reason!r}")

原生 vLLM        :   5896 token/s


ESamp beta=0.0 :   5786 token/s，为原生的 98%；128/128 个样本与原生相同；累计训练 163200 行；eager 原因 ''


ESamp beta=0.5 :   5729 token/s，为原生的 97%；3/128 个样本与原生相同；累计训练 163200 行；eager 原因 ''


Three takeaways from these three lines:

- **Training does not perturb generation**: At `beta = 0`, every request trains ("total trained rows" counts all request-token rows across the five `generate` runs on that engine), yet all 128 samples match plain vLLM word-for-word. Occasionally one or two samples may diverge mid-generation due to floating-point rounding: an engine with a workload extracts intermediate hidden states, altering the compiled graph slightly, so stochastic sampling over 256 tokens can tip across a near-tie;
- **Overhead is tiny**: Running a forward, backward, and AdamW step per token per request drops throughput by only 2–3% compared to plain vLLM, because training runs on a side stream overlapped with the next forward pass;
- **Guidance takes effect**: At `beta = 0.5`, 125 of 128 samples diverge from the baseline, with the small extra cost coming from projecting predictions through the LM head and modifying the sampling distribution each step.

These numbers come from a single run on a local RTX 4090 (Qwen2.5-7B-Instruct, bf16, vLLM 0.32.0, CUDA graphs; 8 prompts × 16 samples = 128 concurrent requests, 256 tokens each; median of 3 runs after 1 warmup), so individual machines and runs vary by a percentage point or two. Benchmarks for the full ESamp workload under the same setup are recorded in [measurements.toml](../reference/measurements.toml): 97.4% of plain vLLM for training only, and 96.3% with training plus guidance.

## What the Framework Handled for You

- **Per-request parameters and optimizer state**: 128 small networks plus their AdamW momentum buffers are paged inside GPU memory budgeted at startup and reclaimed as soon as each request completes;
- **CUDA graphs**: An empty `eager_reason` confirms the model forward, your loss, backward pass, and AdamW updates all run inside CUDA graphs;
- **Overlap**: Training executes on a side stream concurrently with the next forward pass, trading one step of latency—an update computed at step $N$ is visible to `predict` starting at step $N + 2$. For strict step-to-step ordering, set `pllm.ExecutionOptions(publish_lag=0)`;
- **Scheduling**: You never specified which stream training uses or when to synchronize. pLLM's planner infers from your declarations that `loss` writes `distiller` while `predict` reads it, and schedules ordering and overlap automatically.

The full ESamp implementation lives in `pllm.workloads.ESampWorkload`—the same structure as `MiniESamp` here, with gating added to the small network.

## Common Mistakes

Three common mistakes in training workloads: reducing the loss to a single scalar yourself, writing `beta=beta` in the class body, and passing `rows=` to a standalone training method. `testing.check` catches the first two on CPU; the third is rejected at build time.

In [5]:
class ScalarLoss(MiniESamp):
    @pllm.train(state="distiller")
    def loss(self, shallow: torch.Tensor = pllm.hidden(0), deep: torch.Tensor = pllm.hidden(-1),
             f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return (f(shallow) - deep).square().mean()  # Averaged all rows into a single scalar


@dataclass(frozen=True)
class ClassDefault(MiniESamp):
    beta: float = 0.5

    @pllm.guide_sampling(beta=beta, rows="trained")  # Reads the class default 0.5 rather than the instance beta
    def predict(self, shallow: torch.Tensor = pllm.hidden(0), f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return f(shallow)


class SelectedTraining(MiniESamp):
    @pllm.train(state="distiller", rows="trained")  # Trying to train only already-trained requests
    def loss(self, shallow: torch.Tensor = pllm.hidden(0), deep: torch.Tensor = pllm.hidden(-1),
             f: Distiller = pllm.module("distiller")) -> torch.Tensor:
        return (f(shallow) - deep).square().mean(-1)


SMALL = pllm.ModelConfig(MODEL, max_num_seqs=8)
for name, run in (("ScalarLoss", lambda: testing.check(ScalarLoss(0.5), SMALL)),
                  ("ClassDefault", lambda: testing.check(ClassDefault(), SMALL, neutral=ClassDefault(beta=0.0))),
                  ("SelectedTraining", lambda: pllm.explain(SelectedTraining(0.5), SMALL))):
    try:
        run()
    except testing.ContractError as error:
        print(f"{name}: {error.failures[0]}\n")
    except ValueError as error:
        print(f"{name}: {error}\n")

ScalarLoss: [compile] ValueError: training methods return one loss per row; pLLM reduces the selected rows per state
while shape tracing method 'loss'; methods run inside CUDA graphs: use tensor operations such as torch.where, not .item() or Python branches on tensor values

ClassDefault: [neutral] guide_sampling still declares a sampler stage; set its beta to zero.

SelectedTraining: rows= and update_rows= on request training require a model effect without sampler guidance, or @pllm.add_logits; standalone training and @pllm.guide_sampling update every selected state, so remove rows= and update_rows=



**Why `loss` must return one value per row.** How many rows belong to a given request state in a step depends on scheduling: one row during decode, multiple rows during chunked prefill, or rows from multiple requests when sharing state. Only pLLM knows which rows map to which state copy, so pLLM performs the per-state reduction (`reduction="mean"` or `"sum"`). Calling `.mean()` yourself mixes losses across different requests.

**Why `beta=beta` is wrong.** When the class body executes, no instance exists yet, so `beta` resolves to the class default `0.5`—meaning an instance created with `beta=0` still guides sampling. The `neutral=` contract check ("disabling the knob must not alter sampling") catches this immediately. Wrapping the setting in a method (`def strength(self)`) and passing that method to the decorator lets pLLM evaluate it per instance.

**Why standalone training rejects `rows=`.** When training runs alone or alongside `@pllm.guide_sampling`, it uses a fast path designed for "every request learns on every step": each active request state receives one optimizer update per step, publishes a new version, and overlaps with the next forward pass. Skipping updates per row would require conditional version publication, which that path does not implement, so pLLM rejects it at build time instead of silently ignoring your mask. To exclude rows from contributing, filter with `rows=` on the sampling method (as `predict` does here), or switch to `reduction="sum"` and multiply excluded rows by `0` (with the default `"mean"`, zeroed rows still count in the denominator). When paired with `@pllm.add_logits` or model edits (`Adapter`, etc.), training uses conditional updates and `rows=` is supported.

## Self-Check

1. Under default settings, if a training update is computed at step 10, which step is the earliest `predict` can read it? Why is there a delay?

<details><summary>Answer</summary>

Step 12. With the default `publish_lag=1`, training runs on a side stream overlapped with step 11's forward pass, so step 11 cannot read it yet. That one-step lag buys overlap between training and generation; if you need strict sequential visibility, set `pllm.ExecutionOptions(publish_lag=0)` at the cost of no overlap.
</details>

2. At `beta = 0`, every request is still training—why do sampled outputs match plain vLLM?

<details><summary>Answer</summary>

Training only updates the `distiller` state without touching logits, and `guide_sampling` with `beta=0` schedules no sampler work. Both the model forward and sampling match plain vLLM, so tokens match word-for-word (occasional single-sample divergence comes from minor rounding differences between compiled graphs).
</details>

3. What happens to newly arrived requests if you remove `rows="trained"`?

<details><summary>Answer</summary>

For a request that hasn't trained yet, the small network is just an unlearned layer norm whose "prediction" is noise, yet it would still steer sampling by `beta`. `rows="trained"` leaves version-0 requests on standard sampling until their state has been updated at least once.
</details>

Next up, [04 Custom Forward](04_custom_forward.ipynb) takes a different angle: instead of adding side computations, we change which layers the model itself executes.